In [2]:
import torchvision.datasets as dsets
from torch.utils.data import DataLoader
import torchvision.transforms as transforms
import torch
import torch.nn

In [3]:
mnist_train = dsets.MNIST(root="MNIST_data/", train=True, transform=transforms.ToTensor(), download=True)
mnist_test =dsets.MNIST(root="MNIST_data/", train=False, transform=transforms.ToTensor(), download=True)

In [4]:
data_loader = DataLoader(mnist_train, batch_size=100, shuffle=True, drop_last=True)

In [5]:
for X, Y in data_loader:
    X = X.view(-1, 28 * 28) # batchsize, W*H

In [6]:
X.shape

torch.Size([100, 784])

In [7]:
len(data_loader)

600

## Softmax

In [8]:
linear = torch.nn.Linear(784, 10, bias = True).to('cpu')

#parameter
training_epochs = 15
batch_size = 100

#define cost/Loss & Optimizer
criterion = torch.nn.CrossEntropyLoss().to('cpu')
optimizer = torch.optim.SGD(linear.parameters(), lr=0.1)

In [9]:
for epoch in range(training_epochs):
    # X: 이미지, Y: Label
    # reshape input image into (batch_size by 784)
    # Label is not one-hot encoded
    avg_cost = 0
    total_batch = len(data_loader)
    for X, Y in data_loader: #batchsize만큼 data_loader로 학습
        X = X.view(-1, 28*28).to('cpu')
        optimizer.zero_grad()
        hypothesis = linear(X)
        cost = criterion(hypothesis, Y)
        cost.backward() #backward를 이용해서 gradient를 계산
        optimizer.step() #optimizer에 update
        avg_cost += cost / total_batch
    
    print("Epoch: ", "%04d" % (epoch+1), "cost=", "{:.9f}".format(avg_cost))

Epoch:  0001 cost= 0.534929872
Epoch:  0002 cost= 0.359118700
Epoch:  0003 cost= 0.331232816
Epoch:  0004 cost= 0.316506624
Epoch:  0005 cost= 0.307023466
Epoch:  0006 cost= 0.299983472
Epoch:  0007 cost= 0.295221299
Epoch:  0008 cost= 0.290787727
Epoch:  0009 cost= 0.287433058
Epoch:  0010 cost= 0.284315854
Epoch:  0011 cost= 0.282133847
Epoch:  0012 cost= 0.279685766
Epoch:  0013 cost= 0.277677387
Epoch:  0014 cost= 0.276174963
Epoch:  0015 cost= 0.274298966


## Test

In [10]:
with torch.no_grad():
    X_test = mnist_test.test_data.view(-1, 28*28).float().to('cpu')
    Y_test = mnist_test.test_labels.to('cpu')
    
    prediction = linear(X_test)
    correct_prediction = torch.argmax(prediction, 1) == Y_test
    accuracy = correct_prediction.float().mean()
    print("Accuracy: ", accuracy.item())

Accuracy:  0.8828999996185303


/anaconda3/lib/python3.7/site-packages/torchvision/datasets/mnist.py:58: UserWarning: test_data has been renamed data
  warnings.warn("test_data has been renamed data")
/anaconda3/lib/python3.7/site-packages/torchvision/datasets/mnist.py:48: UserWarning: test_labels has been renamed targets
  warnings.warn("test_labels has been renamed targets")


## Epoch, BatchSize, Iteration
#### In the neural network terminology:
    - one epoch: one forward pass and one backward pass of all the training examples
    - batch size: the number of training examples in one forward/backward pass. The higher the batch size, the more memory space you'll need.
    - number of iterations: 배치를 몇번 학습에 사용 하였나? number of passes, each pass using(batch size) number of examples. 
    
### ex. 
1000개의 dataset을 batch size 500으로 나누고 (batch의 개수는 2) 2회 iterations하면 1개의 epoch이 돌았다. 

In [16]:
import matplotlib.pyplot as plt
import random

r = random.randint(0, len(mnist_test) - 1)
X_single_data = mnist_test.test_data[r: r + 1].view(-1, 28*28).float().to('cpu')
Y_single_data = mnist_test.test_labels[r: r + 1].to('cpu')

print('label: ', Y_single_data.item())
single_prediction = linear(X_single_data)

print("Prediction: ", torch.argmax(single_prediction, 1).item())

#plt.imshow(mnist_test.test_data[r:r +1].view(28,28), cmap='Greys', interpolation ='nearest')

plt.show()

label:  4
Prediction:  4
